In [1]:
# import os
# os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/amazon-corretto-11.jdk/Contents/Home"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Bitcoin_Data") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "false") \
    .config("spark.sql.execution.arrow.pyspark.fallback.enabled", "false") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/22 10:28:35 WARN Utils: Your hostname, Assma, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/01/22 10:28:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/22 10:28:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df = spark.read.parquet("../data_silver.parquet")
df.show()

+-------------------+--------+--------+--------+--------+---------+--------------------+------------------+----------------+---------------------+----------------------+------+---------------+--------------------+-----------------+-----------------+-------------------+
|          open_time|    open|    high|     low|   close|   volume|          close_time|quote_asset_volume|number_of_trades|taker_buy_base_volume|taker_buy_quote_volume|ignore|close_t_plus_10|              return|            ma_05|            ma_10|        taker_ratio|
+-------------------+--------+--------+--------+--------+---------+--------------------+------------------+----------------+---------------------+----------------------+------+---------------+--------------------+-----------------+-----------------+-------------------+
|2026-01-20 23:03:00|88296.02|88296.02|88097.15|88097.16| 69.05007|2026-01-20 23:03:...|   6088295.0226226|           10479|             13.79466|       1216135.4714683|     0|       88213.0

In [3]:
df.describe().show()

26/01/22 10:29:24 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+---------------------+----------------------+------+-----------------+--------------------+-----------------+-----------------+--------------------+
|summary|              open|             high|               low|             close|            volume|quote_asset_volume|  number_of_trades|taker_buy_base_volume|taker_buy_quote_volume|ignore|  close_t_plus_10|              return|            ma_05|            ma_10|         taker_ratio|
+-------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+---------------------+----------------------+------+-----------------+--------------------+-----------------+-----------------+--------------------+
|  count|               589|              589|               589|               589|               589|               589|        

In [4]:
df = df.drop("ignore")

In [5]:
df.show()

+-------------------+--------+--------+--------+--------+---------+--------------------+------------------+----------------+---------------------+----------------------+---------------+--------------------+-----------------+-----------------+-------------------+
|          open_time|    open|    high|     low|   close|   volume|          close_time|quote_asset_volume|number_of_trades|taker_buy_base_volume|taker_buy_quote_volume|close_t_plus_10|              return|            ma_05|            ma_10|        taker_ratio|
+-------------------+--------+--------+--------+--------+---------+--------------------+------------------+----------------+---------------------+----------------------+---------------+--------------------+-----------------+-----------------+-------------------+
|2026-01-20 23:03:00|88296.02|88296.02|88097.15|88097.16| 69.05007|2026-01-20 23:03:...|   6088295.0226226|           10479|             13.79466|       1216135.4714683|       88213.03|-0.00225208364454...|88196

In [6]:
gold_path = "data/gold_dataset"

(
    df
    .write
    .mode("overwrite")
    .parquet(gold_path)
)


In [7]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

# Split temporel
def train_test_split_time(df, train_ratio=0.8):
    
    #Split des données selon l'ordre chronologique.
    
    total_rows = df.count()
    train_count = int(total_rows * train_ratio)
    train_df = df.limit(train_count)
    test_df = df.subtract(train_df)
    return train_df, test_df

In [8]:
train_df, test_df = train_test_split_time(df)

In [9]:
# Créer le pipeline ML
def build_pipeline(feature_cols):
    
    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
    gbt = GBTRegressor(featuresCol="features", labelCol="close_t_plus_10", maxIter=100)
    pipeline = Pipeline(stages=[assembler, gbt])
    return pipeline

In [10]:
feature_cols = [
        "open", "high", "low", "close", "volume",
        "quote_asset_volume", "number_of_trades",
        "taker_buy_base_volume", "taker_buy_quote_volume"
    ]

pipeline = build_pipeline(feature_cols)

In [11]:
# Entraîner le modèle
def train_model(pipeline, train_df):
    
    model = pipeline.fit(train_df)
    return model

In [12]:
model = train_model(pipeline, train_df)

In [13]:
from pyspark.ml.evaluation import RegressionEvaluator

def evaluate_model(model, test_df):
    
    # Faire les prédictions
    predictions = model.transform(test_df)
    
    # Afficher les colonnes pertinentes
    predictions.select("close", "close_t_plus_10", "prediction").show(10)
    
    # Définir les évaluateurs
    evaluator_rmse = RegressionEvaluator(
        labelCol="close_t_plus_10", predictionCol="prediction", metricName="rmse"
    )
    evaluator_mae = RegressionEvaluator(
        labelCol="close_t_plus_10", predictionCol="prediction", metricName="mae"
    )
    evaluator_r2 = RegressionEvaluator(
    labelCol="close_t_plus_10", predictionCol="prediction", metricName="r2"
    )

    # Calculer les métriques
    rmse = evaluator_rmse.evaluate(predictions)
    mae = evaluator_mae.evaluate(predictions)
    r2 = evaluator_r2.evaluate(predictions)
    
    # Afficher les métriques
    print(f"RMSE = {rmse}")
    print(f"MAE = {mae}")
    print(f"R² = {r2}")
    
    # Retourner DataFrame des prédictions et métriques
    return predictions, rmse, mae, r2


In [14]:
#  Prédictions et évaluation
predictions, rmse, mae, r2 = evaluate_model(model, test_df)

+--------+---------------+-----------------+
|   close|close_t_plus_10|       prediction|
+--------+---------------+-----------------+
|89262.93|       89261.68|89336.85331160811|
|89628.97|       89207.85|89776.98534175518|
|89159.85|       89189.89|89153.48695138491|
|89228.16|       89118.18|89231.16354222436|
|89208.92|       89161.06|89216.91118767267|
|89374.49|       89286.11|89266.49519042691|
|89606.87|       89592.07|89623.50710078905|
|89730.88|       89606.87| 89645.4981296031|
|89334.22|       89246.01|89436.99514953328|
|89356.48|       89287.27|89485.60222562199|
+--------+---------------+-----------------+
only showing top 10 rows
RMSE = 165.11728212930745
MAE = 128.53282720089487
R² = 0.26648901274069337


In [15]:
def spark_to_xy(df, feature_cols, label_col):
    pdf = df.select(feature_cols + [label_col]).toPandas()
    X = pdf[feature_cols]
    y = pdf[label_col]
    return X, y

In [16]:
label_col = "close_t_plus_10"

X_train, y_train = spark_to_xy(train_df, feature_cols, label_col)
X_test, y_test = spark_to_xy(test_df, feature_cols, label_col)

In [17]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def train_random_forest(X_train, y_train, X_test, y_test):
    
    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1
    )
    
    model.fit(X_train, y_train)
    
    predictions1 = model.predict(X_test)
    
    mae1 = mean_absolute_error(y_test, predictions1)
    rmse1 = np.sqrt(mean_squared_error(y_test, predictions1))
    r21 = r2_score(y_test, predictions1)
    
    print(f"MAE  = {mae1}")
    print(f"RMSE = {rmse1}")
    print(f"R²   = {r21}")
    
    return predictions1, rmse1, mae1, r21


In [18]:
predictions1, rmse1, mae1, r21 = train_random_forest(X_train, y_train, X_test, y_test)

MAE  = 117.97093220204482
RMSE = 152.26208333210846
R²   = 0.3762578453027897


In [19]:
import pandas as pd

results_df = pd.DataFrame({
    "Modèle": ["GBTRegressor", "Random Forest"],
    "MAE": [mae, mae1],
    "RMSE": [rmse, rmse1],
    "R²": [r2, r21]
})


results_df

,Modèle,MAE,RMSE,R²
0,GBTRegressor,128.532827,165.117282,0.266489
1,Random Forest,117.970932,152.262083,0.376258


In [20]:
import matplotlib.pyplot as plt

def plot_two_models(y_test, pred_rf, pred_gbt):

    plt.figure(figsize=(14,6))
    
    # Vraies valeurs
    plt.plot(y_test.values, label="Valeurs réelles", color='blue')
    
    # Prédictions Random Forest
    plt.plot(pred_rf, label="Random Forest", color='red')
    
    # Prédictions GBT
    plt.plot(pred_gbt, label="GBT Regressor", color='green')
    
    plt.title("Comparaison des modèles : Prédictions vs Réel")
    plt.xlabel("Index temporel")
    plt.ylabel("Prix Bitcoin T+10")
    plt.legend()
    plt.show()


In [21]:
# plot_two_models(y_test, predictions1, predictions)